# Simple RAG - Query and Retrieval

**Flow:**

Query → Retrieve → LLM → Answer

## Install Dependencies

In [ ]:
# # Conda environment setup
# !pip install langchain langchain-chroma langchain-openai chromadb

## Basic RAG Code Phase 2 - Query and Retrieval

In [1]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [2]:
# # Colab Store Location
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [3]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [4]:
# Local setup
store_location = "../vector_db/chroma_db1"

In [5]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()

True

### Step 1: Create retriever

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=store_location,
    embedding_function=embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Step 2: Build Modern RAG Chain (LCEL)

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

### Step 3: Compose the chain

In [8]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Step 4: Query

In [9]:
response = rag_chain.invoke("What is this document about?")
print(response)

The document appears to discuss issues related to data privacy and security, particularly in the context of a past incident that exposed personal information such as names, email addresses, and phone numbers. It mentions that the industry addressed and fixed these issues in 2013 and 2014. Additionally, it references a process for users to access their data through Apple's privacy website, indicating a focus on user control over personal information. There is also mention of a financial dispute that does not affect the technical findings related to the data privacy issues, suggesting a discussion of accountability and verification of claims in the context of data security.


## Debug

In [10]:
docs = retriever.invoke("What is this document about?")

for i, d in enumerate(docs):
    print(f'chunk {i}: {d.page_content}')
    print("-" * 50)

chunk 0: was fixed back in 2013 and 2014, 12 years ago.  At the time, many people were exposed.  We fixed that.  We as an industry fixed it.
--------------------------------------------------
chunk 1: site, at least their names and email addresses and probably more.  Sometimes their phone numbers.  Because, hey, there's a chance to win.
--------------------------------------------------
chunk 2: were some reports of that in the UK also, it appears that it's possible to bring up a web page at privacy.apple.com.  You'll be asked to login with your Apple credentials, then respond to a multi-factor prompt on one of your Apple devices.  Once you've done that, you'll be taken to a "Choose the data you wish to download" page.
--------------------------------------------------
chunk 3: "The financial dispute between the parties does not change the technical findings, however, which were verified independently.  It does mean the framing of those findings is contested.  The readers should weigh 